In [1]:
import os
import xarray as xr
import numpy as np

In [2]:
# === Processing functions ===
def adjust_longitude(dataset: xr.Dataset) -> xr.Dataset:
        """Swaps longitude coordinates from range (0, 360) to (-180, 180)
        Args:
            dataset (xr.Dataset): xarray Dataset
        Returns:
            xr.Dataset: xarray Dataset with swapped longitude dimensions
        """
        lon_name = "lon"  # whatever name is in the data

        # Adjust lon values to make sure they are within (-180, 180)
        dataset["_longitude_adjusted"] = xr.where(
            dataset[lon_name] > 180, dataset[lon_name] - 360, dataset[lon_name])
        dataset = (
            dataset.swap_dims({lon_name: "_longitude_adjusted"})
            .sel(**{"_longitude_adjusted": sorted(dataset._longitude_adjusted)})
            .drop_vars(lon_name)
        )

        dataset = dataset.rename({"_longitude_adjusted": lon_name})
        return dataset

In [ ]:
# === Path config ===
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8/"
OBS_DIR = "/glade/work/awells/air_quality/O3_obs/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(8, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        # Load data arrays
        if scenario == "ARISE":
            dates = "2035-2069"
        elif scenario == "SSP245":
            dates = "2020-2069"

        osdma8 = xr.open_dataarray(f"{OSDMA8_DIR}OSDMA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")*10**9  #ppb
        hist = xr.open_dataarray(f"{OSDMA8_DIR}OSDMA8_CESM2_hist_01_1990-2009.nc")*10**9  #ppb
        obs = xr.open_dataset(f"{OBS_DIR}Delang_BME_OSDMA8_1990_2017.nc")["ozone"]

        # Baseline years for fi_2000 and historical
        base = slice("1990", "2009")
        hist_base = hist.sel(year=base).mean("year")
        obs_base = obs.sel(year=base).mean("year")
        obs_base = obs_base.rename({'longitude': 'lon', 'latitude': 'lat'})

        # Define the higher-resolution grid to match observations (0.1 x 0.1)
        new_lat = obs_base['lat']
        new_lon = obs_base['lon']

        # Calculate delta
        delta_fi = adjust_longitude(osdma8 / hist_base)

        # Interpolate to the new grid
        ds_delta_fi = delta_fi.interp(lat=new_lat, lon=new_lon, method='linear')

        # Bias correct delta
        bc_osdma8 = obs_base * ds_delta_fi

        out_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving to {out_path}")
        bc_osdma8.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 08


In [13]:
# === Path config ===
OSDMA8_DIR = "/glade/work/awells/air_quality/CESM/OSDMA8_March/"
OBS_DIR = "/glade/work/awells/air_quality/"
SAVE_DIR = "/glade/work/awells/air_quality/CESM/OSDMA8_BC/"
SCENARIOS = ["ARISE", "SSP245"]


# === Main loop ===
for scenario in SCENARIOS:
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        # Load data arrays
        if scenario == "ARISE":
            dates = "2035-2069"
            periods = [(2035, 2054), (2040, 2059), (2045, 2064), (2050, 2069)]
        elif scenario == "SSP245":
            dates = "2020-2069"
            periods = [(2020, 2039), (2025, 2044), (2030, 2049), (2035, 2054),
                       (2040, 2059), (2045, 2064), (2050, 2069)]

        osdma8 = xr.open_dataarray(f"{OSDMA8_DIR}OSDMA8_CESM2_{scenario}_{ens_num:02d}_{dates}.nc")*10**9  #ppb
        hist = xr.open_dataarray(f"{OSDMA8_DIR}OSDMA8_CESM2_hist_01_1990-2009.nc")*10**9  #ppb
        obs = xr.open_dataset(f"{OBS_DIR}Delang_BME_OSDMA8_1990_2017.nc")["ozone"]

        # Baseline years for fi_2000 and historical
        base = slice("1990", "2009")
        hist_base = hist.sel(year=base).mean("year")
        obs_base = obs.sel(year=base).mean("year")
        obs_base = obs_base.rename({'longitude': 'lon', 'latitude': 'lat'})

        # Define the higher-resolution grid to match observations (0.1 x 0.1)
        new_lat = obs_base['lat']
        new_lon = obs_base['lon']

        # Future periods e.g. (2035, 2054) represents 2045
        for start, end in periods:
            fi = slice(str(start), str(end))
            osdma8_fi = osdma8.sel(year=fi).mean("year")

            # Calculate delta
            delta_fi = adjust_longitude(osdma8_fi / hist_base)

            # Interpolate to the new grid
            ds_delta_fi = delta_fi.interp(lat=new_lat, lon=new_lon, method='linear')

            # Bias correct delta
            bc_osdma8 = obs_base * ds_delta_fi

            out_file = f"OSDMA8_BC_CESM2_{scenario}_{ens_num:02d}_{start}-{end}.nc"
            out_path = os.path.join(SAVE_DIR, out_file)

            print(f"Saving to {out_path}")
            bc_osdma8.to_netcdf(out_path)

print("All processing complete.")

Processing ARISE, Ensemble 01
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_01_2035-2054.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_01_2040-2059.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_01_2045-2064.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_01_2050-2069.nc
Processing ARISE, Ensemble 02
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_02_2035-2054.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_02_2040-2059.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_02_2045-2064.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_02_2050-2069.nc
Processing ARISE, Ensemble 03
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_BC_CESM2_ARISE_03_2035-2054.nc
Saving to /glade/work/awells/air_quality/CESM/OSDMA8_BC/OSDMA8_B